# ML Coursework - Full Project Code Notebook

This notebook contains the complete code from the project scripts and source modules.

Run scripts from terminal as needed:
- python hyperparameter_tuning.py
- python train_model.py
- python generate_report_artifacts.py
- streamlit run app.py

## app.py

In [2]:
from __future__ import annotations
import json
import joblib
import pandas as pd
import streamlit as st
from src.config import BEST_MODEL_FILE, RESULTS_FILE, TARGET_LABELS
from src.data_utils import load_adult_train_test, dataset_profile

st.set_page_config(page_title='Adult Income Predictor Pro', page_icon='💼', layout='wide')

st.markdown('''
<style>
.block-container {padding-top: 1.4rem; padding-bottom: 2rem;}
.hero {padding: 1.2rem 1.4rem; border-radius: 22px; background: linear-gradient(135deg,#0f172a 0%,#1d4ed8 55%,#2563eb 100%); color:white; box-shadow:0 18px 42px rgba(37,99,235,.22); margin-bottom:1rem;}
</style>
''', unsafe_allow_html=True)

@st.cache_data
def get_data():
    train_df, test_df = load_adult_train_test()
    return train_df, test_df, dataset_profile(train_df, test_df)

@st.cache_resource
def get_bundle():
    model = joblib.load(BEST_MODEL_FILE) if BEST_MODEL_FILE.exists() else None
    results = json.loads(RESULTS_FILE.read_text(encoding='utf-8')) if RESULTS_FILE.exists() else {}
    return model, results

train_df, test_df, profile = get_data()
model, results = get_bundle()
full_df = pd.concat([train_df.assign(split='Train'), test_df.assign(split='Test')], ignore_index=True)

if 'history' not in st.session_state:
    st.session_state.history = []

with st.sidebar:
    st.title('Adult Income Predictor Pro')
    page = st.radio('Navigate', ['Dashboard','Data Explorer','Model Lab','Predict'])
    st.caption('Coursework-ready Streamlit software bundle')

st.markdown("<div class='hero'><h2 style='margin:0;'>Adult / Census Income ML Software</h2><p style='margin:.5rem 0 0 0;'>Dataset exploration, model comparison, and live prediction in one polished app.</p></div>", unsafe_allow_html=True)

if page == 'Dashboard':
    c1,c2,c3,c4 = st.columns(4)
    c1.metric('Total rows', f"{profile['rows_total']:,}")
    c2.metric('Train / Test', f"{profile['train_rows']:,} / {profile['test_rows']:,}")
    c3.metric('Features', profile['feature_count'])
    c4.metric('>50K share', f"{profile['positive_rate']*100:.1f}%")
    left,right = st.columns([1.2,1])
    with left:
        st.subheader('Project overview')
        st.write('Use this software in the viva to show the dataset, preprocessing logic, model comparison, and one live prediction. It is intentionally focused and coursework-friendly.')
        if results:
            st.info(f"Deployment model: **{results.get('deployment_model','N/A')}** | Bundle mode: **{results.get('bundle_mode','unknown')}**")
        st.dataframe(full_df.head(12), use_container_width=True)
    with right:
        st.subheader('Missing values')
        missing = profile.get('missing_by_column', {})
        if missing:
            missing_df = pd.DataFrame({'column': list(missing.keys()), 'missing_values': list(missing.values())})
            st.bar_chart(missing_df.set_index('column'))
        st.subheader('Income distribution')
        target_counts = full_df['income'].value_counts().rename_axis('income').reset_index(name='count')
        st.bar_chart(target_counts.set_index('income'))

elif page == 'Data Explorer':
    st.subheader('Explore the data')
    tab1, tab2, tab3 = st.tabs(['Preview','Numeric','Categorical'])
    with tab1:
        split = st.selectbox('Split', ['All','Train','Test'])
        view = full_df if split == 'All' else full_df[full_df['split'] == split]
        st.dataframe(view.head(50), use_container_width=True)
    with tab2:
        num_col = st.selectbox('Numeric feature', ['age','education_num','hours_per_week','capital_gain','capital_loss','fnlwgt'])
        st.bar_chart(full_df.groupby('income')[num_col].mean())
    with tab3:
        cat_col = st.selectbox('Categorical feature', ['education','workclass','marital_status','occupation','relationship','sex','native_country'])
        counts = full_df[cat_col].fillna('Missing').value_counts().head(12).rename_axis(cat_col).reset_index(name='count')
        st.bar_chart(counts.set_index(cat_col))

elif page == 'Model Lab':
    st.subheader('Model comparison')
    if not results:
        st.warning('Run python train_model.py first to generate the bundle.')
    else:
        models_df = pd.DataFrame(results['models']).T.reset_index().rename(columns={'index':'model'})
        cols = [c for c in ['model','accuracy','precision','recall','f1','roc_auc','fit_seconds','note'] if c in models_df.columns]
        st.dataframe(models_df[cols], use_container_width=True)
        st.caption('Deployment is linked to the tuned Random Forest model saved by train_model.py. The table above reflects locally generated metrics.')
        if 'feature_importance' in results:
            st.subheader('Top feature importance')
            fi_df = pd.DataFrame(results['feature_importance'])
            st.bar_chart(fi_df.set_index('feature'))
        cm = results.get('models',{}).get('Random Forest',{}).get('confusion_matrix')
        if cm:
            st.subheader('Random Forest confusion matrix')
            cm_df = pd.DataFrame(cm, index=['Actual <=50K','Actual >50K'], columns=['Pred <=50K','Pred >50K'])
            st.dataframe(cm_df, use_container_width=True)

else:
    st.subheader('Live prediction')
    if model is None:
        st.error('No saved deployment model found. Run python train_model.py first.')
    else:
        col1,col2,col3 = st.columns(3)
        with col1:
            age = st.slider('Age', 18, 90, 37)
            workclass = st.selectbox('Workclass', sorted(train_df['workclass'].dropna().unique().tolist()))
            education = st.selectbox('Education', sorted(train_df['education'].dropna().unique().tolist()))
            education_num = st.slider('Education num', 1, 16, 10)
            marital_status = st.selectbox('Marital status', sorted(train_df['marital_status'].dropna().unique().tolist()))
        with col2:
            occupation = st.selectbox('Occupation', sorted(train_df['occupation'].dropna().unique().tolist()))
            relationship = st.selectbox('Relationship', sorted(train_df['relationship'].dropna().unique().tolist()))
            race = st.selectbox('Race', sorted(train_df['race'].dropna().unique().tolist()))
            sex = st.selectbox('Sex', sorted(train_df['sex'].dropna().unique().tolist()))
            native_country = st.selectbox('Native country', sorted(train_df['native_country'].dropna().unique().tolist()))
        with col3:
            fnlwgt = st.number_input('fnlwgt', min_value=1000, max_value=1500000, value=189778, step=1000)
            capital_gain = st.number_input('Capital gain', min_value=0, max_value=100000, value=0, step=100)
            capital_loss = st.number_input('Capital loss', min_value=0, max_value=5000, value=0, step=10)
            hours_per_week = st.slider('Hours per week', 1, 99, 40)
        if st.button('Predict income class', type='primary'):
            row = pd.DataFrame([{
                'age': age, 'workclass': workclass, 'fnlwgt': fnlwgt, 'education': education, 'education_num': education_num,
                'marital_status': marital_status, 'occupation': occupation, 'relationship': relationship, 'race': race, 'sex': sex,
                'capital_gain': capital_gain, 'capital_loss': capital_loss, 'hours_per_week': hours_per_week, 'native_country': native_country,
            }])
            pred = int(model.predict(row)[0])
            conf = float(model.predict_proba(row)[0][pred])
            label = TARGET_LABELS[pred]
            st.success(f'Predicted income class: **{label}**')
            st.progress(min(max(conf, 0.0), 1.0), text=f'Model confidence: {conf:.1%}')
            hist = row.copy(); hist['prediction'] = label; hist['confidence'] = round(conf,4)
            st.session_state.history.append(hist)
        if st.session_state.history:
            hist_df = pd.concat(st.session_state.history, ignore_index=True)
            st.subheader('Prediction history')
            st.dataframe(hist_df, use_container_width=True)
            st.download_button('Download prediction history', data=hist_df.to_csv(index=False).encode('utf-8'), file_name='prediction_history.csv', mime='text/csv')

2026-04-01 18:24:06.949 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.953 No runtime found, using MemoryCacheStorageManager
2026-04-01 18:24:06.988 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-01 18:24:06.991 Thread 'MainThread':

## train_model.py

In [ ]:
import json
import time
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES, BEST_MODEL_FILE, RESULTS_FILE
from src.data_utils import load_adult_train_test
from src.features import AdultFeatureEngineer, aggregate_feature_importance

train_df, test_df = load_adult_train_test()
X_train = train_df.drop(columns='income')
y_train = (train_df['income'] == '>50K').astype(int)
X_test = test_df.drop(columns='income')
y_test = (test_df['income'] == '>50K').astype(int)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), NUMERIC_FEATURES),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_FEATURES),
])


def evaluate(estimator, X_eval, y_eval):
    pred = estimator.predict(X_eval)
    proba = estimator.predict_proba(X_eval)[:,1]
    return {
        'accuracy': round(float(accuracy_score(y_eval, pred)), 4),
        'precision': round(float(precision_score(y_eval, pred)), 4),
        'recall': round(float(recall_score(y_eval, pred)), 4),
        'f1': round(float(f1_score(y_eval, pred)), 4),
        'roc_auc': round(float(roc_auc_score(y_eval, proba)), 4),
        'confusion_matrix': confusion_matrix(y_eval, pred).tolist(),
    }

# Deployment model: full random forest fit
lr_pipe = Pipeline([
    ('features', AdultFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=3.0, solver='lbfgs', max_iter=1000))
])
start = time.time()
lr_pipe.fit(X_train, y_train)
log_metrics = evaluate(lr_pipe, X_test, y_test)
log_metrics['fit_seconds'] = round(time.time()-start, 2)
log_metrics['best_params'] = {'C':3.0,'solver':'lbfgs','max_iter':1000}

knn_pipe = Pipeline([
    ('features', AdultFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=21, weights='uniform', p=1))
])
start = time.time()
knn_pipe.fit(X_train, y_train)
knn_metrics = evaluate(knn_pipe, X_test, y_test)
knn_metrics['fit_seconds'] = round(time.time()-start, 2)
knn_metrics['best_params'] = {'n_neighbors':21,'weights':'uniform','p':1}

rf_pipe = Pipeline([
    ('features', AdultFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=120, max_depth=None, min_samples_split=10, n_jobs=-1, random_state=42))
])
start = time.time()
rf_pipe.fit(X_train, y_train)
rf_metrics = evaluate(rf_pipe, X_test, y_test)
rf_metrics['fit_seconds'] = round(time.time()-start, 2)
rf_metrics['best_params'] = {'n_estimators':120,'max_depth':None,'min_samples_split':10}

feature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipe.named_steps['model'].feature_importances_
feature_importance = [
    {'feature': feature, 'importance': round(float(score), 4)}
    for feature, score in aggregate_feature_importance(feature_names, importances)[:12]
]

results = {
    'bundle_mode': 'trained-from-scratch-tuned-deployment',
    'deployment_model': 'Random Forest',
    'models': {
        'Logistic Regression': log_metrics,
        'KNN': knn_metrics,
        'Random Forest': rf_metrics,
    },
    'feature_importance': feature_importance,
}

BEST_MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_pipe, BEST_MODEL_FILE)
RESULTS_FILE.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(json.dumps(results, indent=2))

## hyperparameter_tuning.py

In [ ]:
import json
import time
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES, ASSETS_DIR
from src.data_utils import load_adult_train_test
from src.features import AdultFeatureEngineer


train_df, test_df = load_adult_train_test()
X_train = train_df.drop(columns='income')
y_train = (train_df['income'] == '>50K').astype(int)
X_test = test_df.drop(columns='income')
y_test = (test_df['income'] == '>50K').astype(int)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
])


def evaluate(estimator, X_eval, y_eval):
    pred = estimator.predict(X_eval)
    proba = estimator.predict_proba(X_eval)[:, 1]
    return {
        'accuracy': round(float(accuracy_score(y_eval, pred)), 4),
        'precision': round(float(precision_score(y_eval, pred)), 4),
        'recall': round(float(recall_score(y_eval, pred)), 4),
        'f1': round(float(f1_score(y_eval, pred)), 4),
        'roc_auc': round(float(roc_auc_score(y_eval, proba)), 4),
        'confusion_matrix': confusion_matrix(y_eval, pred).tolist(),
    }


def run_search(name, model, param_grid):
    pipe = Pipeline([
        ('features', AdultFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model),
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring='f1',
        cv=3,
        n_jobs=-1,
        verbose=1,
    )

    start = time.time()
    grid.fit(X_train, y_train)
    elapsed = round(time.time() - start, 2)

    best_estimator = grid.best_estimator_
    test_metrics = evaluate(best_estimator, X_test, y_test)

    return {
        'search_type': 'GridSearchCV',
        'cv_folds': 3,
        'optimization_metric': 'f1',
        'candidate_count': len(grid.cv_results_['params']),
        'search_seconds': elapsed,
        'search_space': param_grid,
        'best_params': grid.best_params_,
        'best_cv_f1': round(float(grid.best_score_), 4),
        'test_metrics': test_metrics,
    }


search_results = {
    'dataset': 'UCI Adult Income',
    'target': 'income (>50K vs <=50K)',
    'generated_at': pd.Timestamp.utcnow().isoformat(),
    'models': {}
}

search_results['models']['Logistic Regression'] = run_search(
    'Logistic Regression',
    LogisticRegression(max_iter=1000),
    {
        'model__C': [0.1, 1.0, 3.0],
        'model__solver': ['lbfgs', 'liblinear'],
    }
)

search_results['models']['KNN'] = run_search(
    'KNN',
    KNeighborsClassifier(),
    {
        'model__n_neighbors': [11, 21, 31],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    }
)

search_results['models']['Random Forest'] = run_search(
    'Random Forest',
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {
        'model__n_estimators': [120, 220],
        'model__max_depth': [None, 20],
        'model__min_samples_split': [2, 10],
    }
)

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
out_file = ASSETS_DIR / 'hyperparameter_results.json'
out_file.write_text(json.dumps(search_results, indent=2), encoding='utf-8')

summary = []
for model_name, details in search_results['models'].items():
    summary.append({
        'model': model_name,
        'best_cv_f1': details['best_cv_f1'],
        'test_accuracy': details['test_metrics']['accuracy'],
        'test_precision': details['test_metrics']['precision'],
        'test_recall': details['test_metrics']['recall'],
        'test_f1': details['test_metrics']['f1'],
        'test_roc_auc': details['test_metrics']['roc_auc'],
        'search_seconds': details['search_seconds'],
        'best_params': json.dumps(details['best_params']),
    })

summary_df = pd.DataFrame(summary)
summary_csv = ASSETS_DIR / 'hyperparameter_summary.csv'
summary_df.to_csv(summary_csv, index=False)

print(json.dumps(search_results, indent=2))
print('\nSaved tuning artifacts:')
print(out_file)
print(summary_csv)

## generate_report_artifacts.py

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_utils import load_adult_train_test


ASSETS_DIR = Path("assets")
FIG_DIR = ASSETS_DIR / "figures"
TABLE_DIR = ASSETS_DIR / "tables"
MODEL_RESULTS = ASSETS_DIR / "model_results.json"
HYPERPARAM_SUMMARY = ASSETS_DIR / "hyperparameter_summary.csv"


def ensure_dirs() -> None:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)


def save_eda_artifacts(df: pd.DataFrame) -> None:
    # Missing values bar chart
    missing = df.isna().sum().sort_values(ascending=False)
    missing = missing[missing > 0]
    if not missing.empty:
        plt.figure(figsize=(9, 5))
        sns.barplot(x=missing.index, y=missing.values, color="#1f77b4")
        plt.title("Missing Values by Column")
        plt.ylabel("Count")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "missing_values_by_column.png", dpi=200)
        plt.close()

    # Target distribution bar chart
    target_counts = df["income"].value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    target_df = target_counts.rename_axis("income").reset_index(name="count")
    sns.barplot(data=target_df, x="income", y="count", hue="income", palette=["#2a9d8f", "#e76f51"], legend=False)
    plt.title("Income Class Distribution")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "income_class_distribution.png", dpi=200)
    plt.close()

    # Numeric histograms
    numeric_cols = ["age", "education_num", "hours_per_week"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col], bins=30, ax=axes[i], color="#264653")
        axes[i].set_title(f"Distribution of {col}")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "numeric_feature_histograms.png", dpi=200)
    plt.close(fig)


def save_model_tables() -> None:
    if HYPERPARAM_SUMMARY.exists():
        tuning_df = pd.read_csv(HYPERPARAM_SUMMARY)
        tuning_df = tuning_df.sort_values(by="best_cv_f1", ascending=False)
        tuning_df.to_csv(TABLE_DIR / "tuning_summary_ranked.csv", index=False)

    if MODEL_RESULTS.exists():
        payload = json.loads(MODEL_RESULTS.read_text(encoding="utf-8"))
        rows = []
        for model_name, metrics in payload.get("models", {}).items():
            rows.append(
                {
                    "model": model_name,
                    "accuracy": metrics.get("accuracy"),
                    "precision": metrics.get("precision"),
                    "recall": metrics.get("recall"),
                    "f1": metrics.get("f1"),
                    "roc_auc": metrics.get("roc_auc"),
                    "fit_seconds": metrics.get("fit_seconds"),
                }
            )
        if rows:
            model_df = pd.DataFrame(rows).sort_values(by="f1", ascending=False)
            model_df.to_csv(TABLE_DIR / "model_metrics_ranked.csv", index=False)

            plt.figure(figsize=(8, 4))
            plot_df = model_df.melt(
                id_vars=["model"],
                value_vars=["accuracy", "precision", "recall", "f1"],
                var_name="metric",
                value_name="score",
            )
            sns.barplot(data=plot_df, x="metric", y="score", hue="model")
            plt.title("Model Comparison by Classification Metrics")
            plt.ylim(0.55, 0.90)
            plt.tight_layout()
            plt.savefig(FIG_DIR / "model_comparison_metrics.png", dpi=200)
            plt.close()


def main() -> None:
    ensure_dirs()

    train_df, test_df = load_adult_train_test()
    full_df = pd.concat([train_df.assign(split="Train"), test_df.assign(split="Test")], ignore_index=True)

    save_eda_artifacts(full_df)
    save_model_tables()

    print("Saved report artifacts:")
    print(f"- Figures: {FIG_DIR}")
    print(f"- Tables: {TABLE_DIR}")


if __name__ == "__main__":
    main()

## src/config.py

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
ASSETS_DIR = PROJECT_ROOT / 'assets'
TRAIN_FILE = DATA_DIR / 'adult.data'
TEST_FILE = DATA_DIR / 'adult.test'
BEST_MODEL_FILE = MODELS_DIR / 'best_model.joblib'
RESULTS_FILE = ASSETS_DIR / 'model_results.json'
COLUMNS = ['age','workclass','fnlwgt','education','education_num','marital_status','occupation','relationship','race','sex','capital_gain','capital_loss','hours_per_week','native_country','income']
NUMERIC_FEATURES = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week','capital_net','has_capital_gain','has_capital_loss']
CATEGORICAL_FEATURES = ['workclass','education','marital_status','occupation','relationship','race','sex','native_country','age_band','hours_band']
TARGET_LABELS = {0:'<=50K',1:'>50K'}

## src/data_utils.py

In [ ]:
import pandas as pd
from .config import COLUMNS, TRAIN_FILE, TEST_FILE

def load_adult_train_test():
    train_df = pd.read_csv(TRAIN_FILE, header=None, names=COLUMNS, skipinitialspace=True, na_values='?')
    test_df = pd.read_csv(TEST_FILE, header=None, names=COLUMNS, skipinitialspace=True, na_values='?', comment='|')
    test_df['income'] = test_df['income'].str.replace('.', '', regex=False)
    return train_df, test_df

def dataset_profile(train_df, test_df):
    full_df = pd.concat([train_df.assign(split='Train'), test_df.assign(split='Test')], ignore_index=True)
    missing = full_df.isna().sum().sort_values(ascending=False)
    return {
        'rows_total': int(len(full_df)),
        'train_rows': int(len(train_df)),
        'test_rows': int(len(test_df)),
        'feature_count': 14,
        'positive_rate': float((full_df['income'] == '>50K').mean()),
        'missing_by_column': {k:int(v) for k,v in missing.items() if int(v)>0},
    }

## src/features.py

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from .config import CATEGORICAL_FEATURES

class AdultFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for col in X.select_dtypes(include='object').columns:
            X[col] = X[col].astype(str).str.strip()
        X['capital_net'] = X['capital_gain'] - X['capital_loss']
        X['has_capital_gain'] = (X['capital_gain'] > 0).astype(int)
        X['has_capital_loss'] = (X['capital_loss'] > 0).astype(int)
        X['age_band'] = pd.cut(X['age'], bins=[0,25,35,45,55,65,100], labels=['18-25','26-35','36-45','46-55','56-65','65+'], include_lowest=True).astype(str)
        X['hours_band'] = pd.cut(X['hours_per_week'], bins=[0,25,40,50,100], labels=['Part-time','Standard','Overtime','Heavy'], include_lowest=True).astype(str)
        return X

def aggregate_feature_importance(feature_names, importances):
    grouped = {}
    for raw_name, importance in zip(feature_names, importances):
        base = raw_name
        if raw_name.startswith('num__'):
            base = raw_name.split('num__',1)[1]
        elif raw_name.startswith('cat__'):
            rem = raw_name.split('cat__',1)[1]
            if '__' in rem:
                rem = rem.split('__',1)[1]
            matched = None
            for col in CATEGORICAL_FEATURES:
                if rem == col or rem.startswith(col + '_'):
                    matched = col
                    break
            base = matched or rem
        grouped[base] = grouped.get(base, 0.0) + float(importance)
    return sorted(grouped.items(), key=lambda x: x[1], reverse=True)